# Ligand Preparation GUI for VS Code Jupyter Notebook

This notebook launches a separate Tkinter GUI window for ligand preparation.

**Modes**
- AutoDock Vina / AutoDock-GPU: uses Meeko `mk_prepare_ligand.py`
- AutoDock4: uses MGLTools `pythonsh.exe` + AutoDockTools `prepare_ligand4.py`

Run the setup/install commands in a terminal first, then run the GUI cell.


## Recommended terminal setup

Use Anaconda/Miniconda terminal, not inside the notebook:

```bash
conda create -n ligandprep python=3.10 -y
conda activate ligandprep
conda install -c conda-forge rdkit pandas numpy openbabel -y
pip install meeko
python -m ipykernel install --user --name ligandprep --display-name "Python (ligandprep)"
```

Then open this notebook in VS Code and select kernel: **Python (ligandprep)**.

For AutoDock4, install MGLTools locally and provide these paths in the GUI:

```text
C:\Program Files (x86)\MGLTools-1.5.7\pythonsh.exe
C:\Program Files (x86)\MGLTools-1.5.7\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py
```


In [1]:
%pip install PySide6 rdkit meeko pandas numpy


Note: you may need to restart the kernel to use updated packages.


Final Code

In [ ]:
Final COde

In [1]:
# Ligand Preparation GUI for VS Code / Local Python / VS Code Jupyter
# AutoDock Vina: python.exe -m meeko.cli.mk_prepare_ligand
# AutoDock4: MGLTools python.exe/pythonsh.exe + prepare_ligand4.py + OpenBabel
# Run normally: python ligand_prep_gui_vina_ad4_tabs_live_logs.py
# Run in Jupyter cell: %run ligand_prep_gui_vina_ad4_tabs_live_logs.py

import os
import sys
import re
import csv
import shutil
import zipfile
import traceback
import subprocess
import threading
import tempfile
from pathlib import Path
from datetime import datetime

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, rdMolDescriptors
    from rdkit.Chem.MolStandardize import rdMolStandardize
except Exception as e:
    Chem = None
    RDKit_IMPORT_ERROR = e
else:
    RDKit_IMPORT_ERROR = None

try:
    import tkinter as tk
    from tkinter import ttk, filedialog, messagebox
    from tkinter.scrolledtext import ScrolledText
except Exception as e:
    tk = None
    TK_IMPORT_ERROR = e
else:
    TK_IMPORT_ERROR = None

# ---------------------------
# Your default Windows paths
# ---------------------------
DEFAULT_VINA_MEEKO_PYTHON = r"C:\Users\USER\AppData\Local\Programs\Python\Python313\python.exe"
DEFAULT_OBABEL = r"C:\Program Files\OpenBabel-3.1.1\obabel.exe"
DEFAULT_MGLTOOLS_PYTHON = r"C:\Program Files (x86)\MGLTools-1.5.7\python.exe"
DEFAULT_PREPARE_LIGAND4 = r"C:\Program Files (x86)\MGLTools-1.5.7\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py"

SUPPORTED_STRUCTURE_EXTS = {".sdf", ".mol", ".mol2", ".pdb"}


def safe_filename(name: str, max_len: int = 120) -> str:
    name = str(name).strip() if name is not None else "ligand"
    if not name or name.lower() == "nan":
        name = "ligand"
    name = re.sub(r"[\\/:*?\"<>|]+", "_", name)
    name = re.sub(r"\s+", "_", name)
    name = "".join(ch for ch in name if ch.isalnum() or ch in "._-")
    return name[:max_len] or "ligand"


def which_any(names):
    for n in names:
        p = shutil.which(n)
        if p:
            return p
    return ""


def existing_or_blank(path: str) -> str:
    return path if path and Path(path).exists() else ""


def guess_vina_python() -> str:
    # Prefer user's confirmed working Python 3.13 path; otherwise current Python executable.
    if Path(DEFAULT_VINA_MEEKO_PYTHON).exists():
        return DEFAULT_VINA_MEEKO_PYTHON
    return sys.executable


def guess_obabel() -> str:
    candidates = [
        DEFAULT_OBABEL,
        which_any(["obabel", "obabel.exe"]),
        r"C:\Program Files (x86)\OpenBabel-3.1.1\obabel.exe",
        r"C:\Program Files\OpenBabel-2.4.1\obabel.exe",
        r"C:\Program Files (x86)\OpenBabel-2.4.1\obabel.exe",
        r"C:\Program Files (x86)\MGLTools-1.5.7\OpenBabel-2.3.2\obabel.exe",
    ]
    for c in candidates:
        if c and Path(c).exists():
            return c
    return DEFAULT_OBABEL


def guess_mgl_python() -> str:
    candidates = [
        DEFAULT_MGLTOOLS_PYTHON,
        r"C:\Program Files (x86)\MGLTools-1.5.7\pythonsh.exe",
        r"C:\Program Files\MGLTools-1.5.7\python.exe",
        r"C:\Program Files\MGLTools-1.5.7\pythonsh.exe",
        r"C:\MGLTools-1.5.7\python.exe",
        r"C:\MGLTools-1.5.7\pythonsh.exe",
        which_any(["pythonsh", "pythonsh.exe"]),
    ]
    for c in candidates:
        if c and Path(c).exists():
            return c
    return DEFAULT_MGLTOOLS_PYTHON


def guess_prepare_ligand4() -> str:
    candidates = [
        DEFAULT_PREPARE_LIGAND4,
        r"C:\Program Files\MGLTools-1.5.7\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py",
        r"C:\MGLTools-1.5.7\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py",
        r"C:\Program Files (x86)\PyRx\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py",
        r"C:\Program Files\Chimera 1.18\bin\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py",
    ]
    for c in candidates:
        if c and Path(c).exists():
            return c
    return DEFAULT_PREPARE_LIGAND4


def run_cmd(cmd, timeout=240, cwd=None):
    """Run external command safely without shell. Return ok, stdout, stderr, returncode."""
    try:
        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=timeout,
            shell=False,
            cwd=str(cwd) if cwd else None,
        )
        return result.returncode == 0, result.stdout, result.stderr, result.returncode
    except Exception as e:
        return False, "", str(e), -999


def load_structure_mols(path, obabel_path=""):
    """
    Load SDF/MOL/MOL2/PDB ligand structure files robustly.

    First RDKit is used with sanitize=False so files that contain minor
    valence/typing issues can still be selected and loaded. Sanitization and
    standardization are performed later by the normal preparation pipeline.

    If RDKit cannot read the structure and OpenBabel is available, the input
    is converted temporarily to SDF and read again. No converted structure
    file is kept in the final output folder.
    """
    path = Path(path)
    ext = path.suffix.lower()
    mols = []

    try:
        if ext == ".sdf":
            suppl = Chem.SDMolSupplier(
                str(path),
                removeHs=False,
                sanitize=False,
                strictParsing=False,
            )
            mols = [m for m in suppl if m is not None]

        elif ext == ".mol":
            m = Chem.MolFromMolFile(
                str(path),
                removeHs=False,
                sanitize=False,
                strictParsing=False,
            )
            mols = [m] if m is not None else []

        elif ext == ".mol2":
            m = Chem.MolFromMol2File(
                str(path),
                removeHs=False,
                sanitize=False,
                cleanupSubstructures=True,
            )
            mols = [m] if m is not None else []

        elif ext == ".pdb":
            m = Chem.MolFromPDBFile(
                str(path),
                removeHs=False,
                sanitize=False,
                proximityBonding=True,
            )
            mols = [m] if m is not None else []
    except Exception:
        mols = []

    if mols:
        return mols

    # Optional OpenBabel fallback for structure files RDKit cannot parse directly.
    obabel = str(obabel_path or "").strip()
    if obabel and Path(obabel).exists() and ext in SUPPORTED_STRUCTURE_EXTS:
        try:
            with tempfile.TemporaryDirectory(prefix="ligand_read_") as td:
                converted = Path(td) / f"{safe_filename(path.stem)}_converted.sdf"
                ok, out, err, code = run_cmd(
                    [obabel, str(path), "-O", str(converted)],
                    timeout=120,
                )
                if ok and converted.exists() and converted.stat().st_size > 0:
                    suppl = Chem.SDMolSupplier(
                        str(converted),
                        removeHs=False,
                        sanitize=False,
                        strictParsing=False,
                    )
                    mols = [m for m in suppl if m is not None]
                    if mols:
                        return mols
        except Exception:
            pass

    return []


def canonical_smiles(mol):
    try:
        m = Chem.RemoveHs(Chem.Mol(mol), sanitize=False)
        Chem.SanitizeMol(m)
        return Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)
    except Exception:
        try:
            return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        except Exception:
            return ""


def undefined_stereo_count(mol):
    try:
        chiral = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
        return sum(1 for _, label in chiral if label == "?")
    except Exception:
        return 0


def formal_charge(mol):
    try:
        return int(sum(a.GetFormalCharge() for a in mol.GetAtoms()))
    except Exception:
        return 0


def validate_pdbqt(path):
    try:
        p = Path(path)
        if not p.exists() or p.stat().st_size < 80:
            return False, "PDBQT missing or too small"
        txt = p.read_text(errors="ignore")
        if "ATOM" not in txt and "HETATM" not in txt:
            return False, "PDBQT lacks ATOM/HETATM records"
        if "TORSDOF" not in txt:
            return False, "PDBQT lacks TORSDOF record"
        return True, "OK"
    except Exception as e:
        return False, str(e)


class LigandPreparator:
    def __init__(self, settings, log_func=None, progress_func=None, counters_func=None):
        self.settings = settings
        self.log = log_func or print
        self.progress = progress_func or (lambda current, total, name="": None)
        self.counters = counters_func or (lambda success, failed, dup: None)

        self.out_dir = Path(settings["output_dir"]).resolve()
        self.out_dir.mkdir(parents=True, exist_ok=True)

        # Final user-visible outputs only:
        #   output_dir/PDBQT/*.pdbqt
        #   output_dir/ligands.txt
        self.pdbqt_dir = self.out_dir / "PDBQT"
        self.pdbqt_dir.mkdir(parents=True, exist_ok=True)

        # SDF/MOL2 preparation files remain temporary and are deleted automatically.
        self._work_root = Path(tempfile.mkdtemp(prefix="ligand_prep_work_"))
        self.sdf_dir = self._work_root / "prepared_3D_SDF"
        self.intermediate_dir = self._work_root / "intermediate_MOL2"
        self.sdf_dir.mkdir(parents=True, exist_ok=True)
        self.intermediate_dir.mkdir(parents=True, exist_ok=True)

        self.rows = []
        self.skipped_duplicates = []
        self.seen_smiles = set()
        self.success_count = 0
        self.failed_count = 0

    def load_inputs(self):
        mode = self.settings["input_mode"]
        records = []

        if mode == "SMILES text":
            raw = self.settings.get("smiles_text", "")
            items = []
            for line in raw.replace(";", "\n").splitlines():
                line = line.strip()
                if not line:
                    continue
                if "," in line:
                    smi, name = line.split(",", 1)
                else:
                    parts = line.split()
                    smi = parts[0]
                    name = "_".join(parts[1:]) if len(parts) > 1 else f"ligand_{len(items)+1}"
                items.append((smi.strip(), name.strip()))
            for smi, name in items:
                records.append({"name": name, "smiles": smi, "mol": None, "source": "SMILES text"})

        elif mode == "CSV with SMILES":
            if pd is None:
                raise RuntimeError("pandas is not installed. Install pandas first.")
            path = Path(self.settings["input_path"])
            df = pd.read_csv(path)
            smiles_col = self.settings.get("smiles_col", "").strip()
            name_col = self.settings.get("name_col", "").strip()
            if not smiles_col:
                for c in df.columns:
                    if c.lower().strip() in ["smiles", "smi", "canonical_smiles", "canonical smiles"]:
                        smiles_col = c
                        break
            if not smiles_col or smiles_col not in df.columns:
                raise RuntimeError(f"SMILES column not found. Available columns: {list(df.columns)}")
            if name_col and name_col not in df.columns:
                self.log(f"Name column '{name_col}' not found. Generated names will be used.")
                name_col = ""
            for idx, row in df.iterrows():
                smi = str(row[smiles_col]).strip()
                if not smi or smi.lower() == "nan":
                    continue
                name = str(row[name_col]).strip() if name_col else f"ligand_{idx+1}"
                records.append({"name": name, "smiles": smi, "mol": None, "source": str(path.name)})

        elif mode == "Structure files":
            paths = [Path(p) for p in self.settings.get("input_files", []) if p]
            obabel = self.settings.get("obabel_path", "").strip()

            for fp in paths:
                if fp.suffix.lower() not in SUPPORTED_STRUCTURE_EXTS:
                    self.log(f"  Ignored unsupported file: {fp.name}")
                    continue
                if not fp.exists():
                    self.log(f"  File not found: {fp}")
                    continue

                mols = load_structure_mols(fp, obabel_path=obabel)
                if not mols:
                    self.log(f"  Could not read structure file: {fp.name}")
                    continue

                # For a normal one-ligand-per-file workflow, the ligand name is
                # exactly the selected file name without its extension.
                # If an SDF contains multiple molecules, numbered suffixes are used.
                for i, mol in enumerate(mols, 1):
                    name = fp.stem if len(mols) == 1 else f"{fp.stem}_{i}"
                    records.append({
                        "name": name,
                        "smiles": "",
                        "mol": mol,
                        "source": fp.name,
                    })

        elif mode == "Folder of structures":
            folder = Path(self.settings["input_path"])
            obabel = self.settings.get("obabel_path", "").strip()
            paths = [
                p for p in folder.iterdir()
                if p.is_file() and p.suffix.lower() in SUPPORTED_STRUCTURE_EXTS
            ]

            for fp in sorted(paths, key=lambda p: p.name.lower()):
                mols = load_structure_mols(fp, obabel_path=obabel)
                if not mols:
                    self.log(f"  Could not read structure file: {fp.name}")
                    continue

                for i, mol in enumerate(mols, 1):
                    name = fp.stem if len(mols) == 1 else f"{fp.stem}_{i}"
                    records.append({
                        "name": name,
                        "smiles": "",
                        "mol": mol,
                        "source": fp.name,
                    })
        else:
            raise RuntimeError("Unknown input mode")
        return records

    def standardize_mol(self, rec):
        name = safe_filename(rec["name"])
        source_smiles = rec.get("smiles", "")
        mol = rec.get("mol")

        if mol is None:
            mol = Chem.MolFromSmiles(source_smiles)
            if mol is None:
                raise ValueError("Invalid SMILES")
        else:
            mol = Chem.Mol(mol)

        try:
            Chem.SanitizeMol(mol)
        except Exception:
            mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(mol, catchErrors=True)

        try:
            mol = rdMolStandardize.Cleanup(mol)
        except Exception:
            pass

        if self.settings.get("remove_salts", True):
            try:
                mol = rdMolStandardize.FragmentParent(mol)
            except Exception:
                frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
                if frags:
                    mol = max(frags, key=lambda m: m.GetNumHeavyAtoms())

        if self.settings.get("canonical_tautomer", True):
            try:
                enumerator = rdMolStandardize.TautomerEnumerator()
                mol = enumerator.Canonicalize(mol)
            except Exception:
                pass

        can_smi = canonical_smiles(mol)
        if self.settings.get("deduplicate", True) and can_smi:
            if can_smi in self.seen_smiles:
                self.skipped_duplicates.append({"name": name, "canonical_smiles": can_smi, "source": rec.get("source", "")})
                raise RuntimeError("Duplicate canonical SMILES skipped")
            self.seen_smiles.add(can_smi)

        mol.SetProp("_Name", name)
        mol.SetProp("OriginalName", str(rec["name"]))
        mol.SetProp("Source", str(rec.get("source", "")))
        mol.SetProp("Canonical_SMILES", can_smi)
        return mol, name, can_smi

    def generate_3d(self, mol):
        ff_choice = self.settings.get("force_field", "AUTO: MMFF94 then UFF")
        n_confs = max(1, int(self.settings.get("num_conformers", 20)))
        mol_h = Chem.AddHs(mol, addCoords=True)

        params = AllChem.ETKDGv3()
        params.randomSeed = int(self.settings.get("random_seed", 42))
        params.pruneRmsThresh = 0.5
        params.useSmallRingTorsions = True
        params.useMacrocycleTorsions = True

        conf_ids = list(AllChem.EmbedMultipleConfs(mol_h, numConfs=n_confs, params=params))
        if not conf_ids:
            cid = AllChem.EmbedMolecule(mol_h, randomSeed=int(self.settings.get("random_seed", 42)), useRandomCoords=True)
            if cid < 0:
                raise RuntimeError("3D conformer generation failed")
            conf_ids = [cid]

        if ff_choice == "ETKDG only; no minimization":
            best_cid = conf_ids[0]
            used_ff = "ETKDG only"
        else:
            best_cid, used_ff = self._minimize_and_pick_best(mol_h, conf_ids, ff_choice)

        best = Chem.Mol(mol_h)
        conf = Chem.Conformer(mol_h.GetConformer(best_cid))
        best.RemoveAllConformers()
        best.AddConformer(conf, assignId=True)
        best.SetProp("ForceFieldUsed", used_ff)
        return best, used_ff, len(conf_ids)

    def _minimize_and_pick_best(self, mol, conf_ids, ff_choice):
        if ff_choice.startswith("AUTO"):
            requested = ["MMFF94", "UFF"]
        elif ff_choice in ["MMFF94", "MMFF94s", "UFF"]:
            requested = [ff_choice]
        else:
            requested = ["MMFF94", "UFF"]

        last_error = None
        for ff in requested:
            energies = []
            try:
                if ff in ["MMFF94", "MMFF94s"]:
                    variant = "MMFF94s" if ff == "MMFF94s" else "MMFF94"
                    props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant=variant)
                    if props is None:
                        raise RuntimeError(f"{ff} parameters unavailable")
                    for cid in conf_ids:
                        try:
                            AllChem.MMFFOptimizeMolecule(mol, mmffVariant=variant, confId=cid, maxIters=1000)
                            force_field = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
                            energies.append((force_field.CalcEnergy(), cid))
                        except Exception as e:
                            last_error = e
                elif ff == "UFF":
                    for cid in conf_ids:
                        try:
                            AllChem.UFFOptimizeMolecule(mol, confId=cid, maxIters=1000)
                            force_field = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
                            energies.append((force_field.CalcEnergy(), cid))
                        except Exception as e:
                            last_error = e
                if energies:
                    energies.sort(key=lambda x: x[0])
                    return energies[0][1], ff
            except Exception as e:
                last_error = e
                continue
        raise RuntimeError(f"Force-field minimization failed: {last_error}")

    def write_sdf(self, mol3d, name):
        sdf_path = self.sdf_dir / f"{name}.sdf"
        writer = Chem.SDWriter(str(sdf_path))
        writer.write(mol3d)
        writer.close()
        return sdf_path

    def prepare_vina_pdbqt(self, sdf_path, name):
        vina_python = self.settings.get("vina_python", "").strip() or guess_vina_python()
        if not Path(vina_python).exists():
            raise RuntimeError("Vina Meeko Python.exe path is missing or invalid")
        pdbqt_path = self.pdbqt_dir / f"{name}.pdbqt"
        cmd = [vina_python, "-m", "meeko.cli.mk_prepare_ligand", "-i", str(sdf_path), "-o", str(pdbqt_path)]
        ok, out, err, code = run_cmd(cmd, timeout=int(self.settings.get("timeout_sec", 240)))
        if not ok:
            raise RuntimeError(f"Meeko module failed | code={code} | stderr={err.strip()} | stdout={out.strip()}")
        valid, msg = validate_pdbqt(pdbqt_path)
        if not valid:
            raise RuntimeError(f"Invalid PDBQT from Meeko: {msg}")
        return pdbqt_path, "Meeko module: python.exe -m meeko.cli.mk_prepare_ligand"

    def sdf_to_mol2_with_obabel(self, sdf_path, name):
        obabel = self.settings.get("obabel_path", "").strip() or guess_obabel()
        if not Path(obabel).exists():
            raise RuntimeError("OpenBabel obabel.exe path is required for AutoDock4 SDF to MOL2 conversion")
        mol2_path = self.intermediate_dir / f"{name}.mol2"
        cmd = [obabel, str(sdf_path), "-O", str(mol2_path)]
        ok, out, err, code = run_cmd(cmd, timeout=int(self.settings.get("timeout_sec", 240)))
        if not ok or not mol2_path.exists() or mol2_path.stat().st_size < 80:
            raise RuntimeError(f"OpenBabel SDF to MOL2 failed | code={code} | stderr={err.strip()} | stdout={out.strip()}")
        return mol2_path

    def prepare_ad4_pdbqt(self, sdf_path, name):
        mgl_python = self.settings.get("mgl_python", "").strip()
        prepare_ligand4 = self.settings.get("prepare_ligand4_path", "").strip()
        if not mgl_python or not Path(mgl_python).exists():
            raise RuntimeError("MGLTools python.exe/pythonsh.exe path is missing or invalid")
        if not prepare_ligand4 or not Path(prepare_ligand4).exists():
            raise RuntimeError("prepare_ligand4.py path is missing or invalid")

        # First make a real MOL2 file using OpenBabel.
        mol2_path = self.sdf_to_mol2_with_obabel(sdf_path, name)
        final_pdbqt_path = self.pdbqt_dir / f"{name}.pdbqt"

        # IMPORTANT FIX FOR WINDOWS/MGLTOOLS:
        # AutoDockTools/MolKit sometimes fails when -l/-o are long absolute Windows paths
        # or when output folders contain spaces. Therefore we run prepare_ligand4.py
        # inside a short temporary working folder and pass only local filenames.
        # Keep AutoDock4 working files temporary. Final output remains only PDBQT.
        ad4_temp_root = self._work_root / "AD4_LIGPREP_TEMP"
        lig_work = ad4_temp_root / safe_filename(name)
        lig_work.mkdir(parents=True, exist_ok=True)

        local_mol2 = lig_work / f"{name}.mol2"
        local_pdbqt = lig_work / f"{name}.pdbqt"
        shutil.copy2(mol2_path, local_mol2)

        # -A hydrogens: add hydrogens if missing
        # -U nphs_lps: merge nonpolar hydrogens and lone pairs; standard AD4 ligand cleanup
        cmd = [
            mgl_python,
            prepare_ligand4,
            "-l", local_mol2.name,
            "-o", local_pdbqt.name,
            "-A", "hydrogens",
            "-U", "nphs_lps",
        ]
        ok, out, err, code = run_cmd(
            cmd,
            timeout=int(self.settings.get("timeout_sec", 240)),
            cwd=lig_work,
        )
        if not ok:
            raise RuntimeError(
                f"prepare_ligand4.py failed | code={code} | working_dir={lig_work} | "
                f"stderr={err.strip()} | stdout={out.strip()}"
            )
        if not local_pdbqt.exists() or local_pdbqt.stat().st_size < 80:
            raise RuntimeError(f"prepare_ligand4.py ran but did not create PDBQT in {lig_work}")

        shutil.copy2(local_pdbqt, final_pdbqt_path)
        valid, msg = validate_pdbqt(final_pdbqt_path)
        if not valid:
            raise RuntimeError(f"Invalid PDBQT from prepare_ligand4.py: {msg}")
        return final_pdbqt_path, "AutoDockTools prepare_ligand4.py"

    def process_all(self):
        if Chem is None:
            raise RuntimeError(f"RDKit import failed: {RDKit_IMPORT_ERROR}")
        records = self.load_inputs()
        total = len(records)
        if total == 0:
            raise RuntimeError("No valid input molecule found")

        self.log(f"Loaded {total} input molecule(s).")
        mode = self.settings.get("docking_mode", "AutoDock Vina")

        for i, rec in enumerate(records, 1):
            base_name = safe_filename(rec.get("name", f"ligand_{i}"))
            self.progress(i - 1, total, base_name)
            self.counters(self.success_count, self.failed_count, len(self.skipped_duplicates))
            self.log(f"\n[{i}/{total}] Processing: {base_name}")
            row = {
                "input_index": i,
                "name": base_name,
                "source": rec.get("source", ""),
                "docking_mode": mode,
                "status": "FAILED",
                "reason": "",
                "canonical_smiles": "",
                "formal_charge": "",
                "rotatable_bonds": "",
                "heavy_atoms": "",
                "undefined_stereo_centers": "",
                "force_field_used": "",
                "conformers_generated": "",
                "sdf_path": "",
                "pdbqt_path": "",
                "method": "",
            }
            try:
                mol, name, can_smi = self.standardize_mol(rec)
                row["name"] = name
                row["canonical_smiles"] = can_smi
                row["formal_charge"] = formal_charge(mol)
                row["rotatable_bonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)
                row["heavy_atoms"] = mol.GetNumHeavyAtoms()
                row["undefined_stereo_centers"] = undefined_stereo_count(mol)

                if row["undefined_stereo_centers"]:
                    self.log(f"  Warning: {row['undefined_stereo_centers']} undefined stereocenter(s).")
                if row["rotatable_bonds"] != "" and int(row["rotatable_bonds"]) > 15:
                    self.log(f"  Warning: high rotatable bonds = {row['rotatable_bonds']}.")

                mol3d, used_ff, n_confs = self.generate_3d(mol)
                row["force_field_used"] = used_ff
                row["conformers_generated"] = n_confs
                sdf_path = self.write_sdf(mol3d, name)
                row["sdf_path"] = str(sdf_path)
                self.log(f"  Temporary 3D SDF prepared: {sdf_path.name} | FF={used_ff} | conformers={n_confs}")

                if mode == "AutoDock Vina":
                    pdbqt_path, method = self.prepare_vina_pdbqt(sdf_path, name)
                else:
                    pdbqt_path, method = self.prepare_ad4_pdbqt(sdf_path, name)

                row["pdbqt_path"] = str(pdbqt_path)
                row["method"] = method
                row["status"] = "SUCCESS"
                row["reason"] = "OK"
                self.success_count += 1
                self.log(f"  SUCCESS: {pdbqt_path.name}")
            except Exception as e:
                reason = str(e)
                row["reason"] = reason
                if "Duplicate canonical SMILES skipped" in reason:
                    row["status"] = "SKIPPED_DUPLICATE"
                    self.log(f"  SKIPPED DUPLICATE: {base_name}")
                else:
                    row["status"] = "FAILED"
                    self.failed_count += 1
                    self.log(f"  FAILED: {reason}")
            self.rows.append(row)
            self.progress(i, total, base_name)
            self.counters(self.success_count, self.failed_count, len(self.skipped_duplicates))

        try:
            return self.write_reports()
        finally:
            # Remove temporary prepared SDF/MOL2/AutoDock4 working files.
            shutil.rmtree(self._work_root, ignore_errors=True)

    def write_reports(self):
        success_rows = [r for r in self.rows if r["status"] == "SUCCESS"]
        failed_rows = [r for r in self.rows if r["status"] == "FAILED"]
        skipped_rows = [r for r in self.rows if r["status"] == "SKIPPED_DUPLICATE"]

        # User requested only ligand names in ligands.txt and final PDBQT files.
        ligand_list = self.out_dir / "ligands.txt"
        with open(ligand_list, "w", encoding="utf-8") as f:
            for r in success_rows:
                f.write(f"{r['name']}\n")

        self.log("\n==============================")
        self.log("PROCESSING COMPLETE")
        self.log(f"Successful PDBQT: {len(success_rows)}")
        self.log(f"Failed: {len(failed_rows)}")
        self.log(f"Skipped duplicate: {len(skipped_rows)}")
        self.log(f"PDBQT folder: {self.pdbqt_dir}")
        self.log(f"Ligand names TXT: {ligand_list}")
        if failed_rows:
            self.log("Failed ligand details are available in the live log.")
        self.log("==============================")

        return {
            "total": len(self.rows),
            "success": len(success_rows),
            "failed": len(failed_rows),
            "duplicates": len(skipped_rows),
            "pdbqt_dir": str(self.pdbqt_dir),
            "ligands_txt": str(ligand_list),
        }



class LigandPrepGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Ligand Preparation GUI - AutoDock Vina / AutoDock4")
        self.root.geometry("1220x820")
        self.root.minsize(1000, 680)

        self.docking_mode = tk.StringVar(value="AutoDock Vina")
        self.input_mode = tk.StringVar(value="CSV with SMILES")
        self.force_field = tk.StringVar(value="MMFF94")
        self.input_path = tk.StringVar(value="")
        self.input_files = []
        self.output_dir = tk.StringVar(value=str(Path.cwd() / "ligand_prep_output"))
        self.smiles_col = tk.StringVar(value="smiles")
        self.name_col = tk.StringVar(value="Compound_name")
        self.smiles_text = tk.StringVar(value="")
        self.vina_python = tk.StringVar(value=guess_vina_python())
        self.obabel_path = tk.StringVar(value=guess_obabel())
        self.mgl_python = tk.StringVar(value=guess_mgl_python())
        self.prepare_ligand4_path = tk.StringVar(value=guess_prepare_ligand4())
        self.remove_salts = tk.BooleanVar(value=True)
        self.canonical_tautomer = tk.BooleanVar(value=True)
        self.deduplicate = tk.BooleanVar(value=True)
        self.num_conformers = tk.IntVar(value=20)
        self.random_seed = tk.IntVar(value=42)
        self.timeout_sec = tk.IntVar(value=240)

        self.is_running = False
        self.log_buffer = []
        self.log_widgets = []
        self.log_window = None

        self.total_var = tk.StringVar(value="0")
        self.current_var = tk.StringVar(value="0")
        self.success_var = tk.StringVar(value="0")
        self.failed_var = tk.StringVar(value="0")
        self.duplicate_var = tk.StringVar(value="0")
        self.current_name_var = tk.StringVar(value="Ready")

        self._build_ui()
        self._update_input_visibility()
        self._update_mode_visibility()
        self.log("Ready. Select mode, input ligands, tool paths, and click RUN.")
        self.log("Live logs are available in the 'Live Logs' tab or by clicking 'Open live log window'.")
        self.log("For Vina: use Python.exe with Meeko installed. For AutoDock4: MGLTools python.exe + prepare_ligand4.py + obabel.exe are required.")

    def _build_ui(self):
        style = ttk.Style()
        try:
            style.theme_use("clam")
        except Exception:
            pass

        outer = ttk.Frame(self.root, padding=8)
        outer.pack(fill=tk.BOTH, expand=True)

        self.nb = ttk.Notebook(outer)
        self.nb.pack(fill=tk.BOTH, expand=True)

        self.tab_setup = ttk.Frame(self.nb, padding=8)
        self.tab_tools = ttk.Frame(self.nb, padding=8)
        self.tab_logs = ttk.Frame(self.nb, padding=8)
        self.tab_reports = ttk.Frame(self.nb, padding=8)

        self.nb.add(self.tab_setup, text="1 Setup")
        self.nb.add(self.tab_tools, text="2 Tool paths")
        self.nb.add(self.tab_logs, text="3 Live Logs / Progress")
        self.nb.add(self.tab_reports, text="4 Output")

        self._build_setup_tab()
        self._build_tools_tab()
        self._build_logs_tab()
        self._build_reports_tab()
        self._build_bottom_bar(outer)

    def _build_setup_tab(self):
        frame = self.tab_setup

        mode_box = ttk.LabelFrame(frame, text="1) Select docking preparation mode", padding=10)
        mode_box.pack(fill=tk.X, pady=5)
        ttk.Radiobutton(mode_box, text="AutoDock Vina / Vina 1.2 ligand PDBQT using Meeko module", variable=self.docking_mode, value="AutoDock Vina", command=self._update_mode_visibility).pack(anchor="w", pady=2)
        ttk.Radiobutton(mode_box, text="AutoDock4 ligand PDBQT using MGLTools AutoDockTools prepare_ligand4.py", variable=self.docking_mode, value="AutoDock4", command=self._update_mode_visibility).pack(anchor="w", pady=2)

        input_box = ttk.LabelFrame(frame, text="2) Input ligands", padding=10)
        input_box.pack(fill=tk.X, pady=5)
        row0 = ttk.Frame(input_box)
        row0.pack(fill=tk.X, pady=3)
        ttk.Label(row0, text="Input type:", width=16).pack(side=tk.LEFT)
        combo = ttk.Combobox(row0, textvariable=self.input_mode, values=["CSV with SMILES", "SMILES text", "Structure files", "Folder of structures"], state="readonly", width=30)
        combo.pack(side=tk.LEFT, padx=5)
        combo.bind("<<ComboboxSelected>>", lambda e: self._update_input_visibility())

        self.input_path_row = ttk.Frame(input_box)
        self.input_path_row.pack(fill=tk.X, pady=3)
        ttk.Label(self.input_path_row, text="Path:", width=16).pack(side=tk.LEFT)
        ttk.Entry(self.input_path_row, textvariable=self.input_path).pack(side=tk.LEFT, fill=tk.X, expand=True, padx=5)
        ttk.Button(self.input_path_row, text="Browse", command=self._browse_input_path).pack(side=tk.LEFT)

        self.csv_row = ttk.Frame(input_box)
        self.csv_row.pack(fill=tk.X, pady=3)
        ttk.Label(self.csv_row, text="SMILES column:", width=16).pack(side=tk.LEFT)
        ttk.Entry(self.csv_row, textvariable=self.smiles_col, width=22).pack(side=tk.LEFT, padx=5)
        ttk.Label(self.csv_row, text="Name/ID column:").pack(side=tk.LEFT, padx=(10, 0))
        ttk.Entry(self.csv_row, textvariable=self.name_col, width=25).pack(side=tk.LEFT, padx=5)
        ttk.Label(self.csv_row, text="Leave blank if absent; generated names will be used.").pack(side=tk.LEFT, padx=8)

        self.smiles_row = ttk.Frame(input_box)
        self.smiles_row.pack(fill=tk.X, pady=3)
        ttk.Label(self.smiles_row, text="SMILES text:", width=16).pack(side=tk.LEFT)
        ttk.Entry(self.smiles_row, textvariable=self.smiles_text).pack(side=tk.LEFT, fill=tk.X, expand=True, padx=5)
        ttk.Label(self.smiles_row, text="Format: SMILES or SMILES,Name").pack(side=tk.LEFT, padx=5)

        prep_box = ttk.LabelFrame(frame, text="3) Standardization and 3D generation", padding=10)
        prep_box.pack(fill=tk.X, pady=5)
        row = ttk.Frame(prep_box)
        row.pack(fill=tk.X, pady=3)
        ttk.Label(row, text="Force field:").pack(side=tk.LEFT)
        ttk.Combobox(row, textvariable=self.force_field, values=["MMFF94", "MMFF94s", "UFF", "AUTO: MMFF94 then UFF", "ETKDG only; no minimization"], state="readonly", width=28).pack(side=tk.LEFT, padx=5)
        ttk.Label(row, text="No. conformers:").pack(side=tk.LEFT, padx=(20, 2))
        ttk.Spinbox(row, from_=1, to=200, textvariable=self.num_conformers, width=8).pack(side=tk.LEFT)
        ttk.Label(row, text="Random seed:").pack(side=tk.LEFT, padx=(20, 2))
        ttk.Spinbox(row, from_=1, to=999999, textvariable=self.random_seed, width=10).pack(side=tk.LEFT)

        ttk.Label(prep_box, text="Meaning of tick boxes: checked/ticked = ON and option will be applied; empty/unchecked = OFF.", foreground="#004c99").pack(anchor="w", pady=(8, 2))
        ttk.Checkbutton(prep_box, text="Remove salts/counterions; keep parent fragment  (ON = remove Na+, Cl-, solvent fragments and keep main ligand)", variable=self.remove_salts).pack(anchor="w", pady=3)
        ttk.Checkbutton(prep_box, text="Canonical tautomer by RDKit  (ON = standardize to one canonical tautomer; not full pH-specific protonation)", variable=self.canonical_tautomer).pack(anchor="w", pady=3)
        ttk.Checkbutton(prep_box, text="Skip duplicate canonical SMILES  (ON = same compound prepared once; duplicates written in skipped_duplicates.txt)", variable=self.deduplicate).pack(anchor="w", pady=3)

    def _build_tools_tab(self):
        frame = self.tab_tools
        tools_box = ttk.LabelFrame(frame, text="Tool paths - default paths are pre-filled; browse to change if needed", padding=10)
        tools_box.pack(fill=tk.BOTH, expand=True, pady=5)

        self.vina_row = self._path_row(tools_box, "Vina Meeko Python.exe:", self.vina_python, self._browse_vina_python, 0, "Used as: python.exe -m meeko.cli.mk_prepare_ligand")
        self.obabel_row = self._path_row(tools_box, "OpenBabel obabel.exe:", self.obabel_path, self._browse_obabel, 1, "Required for AutoDock4 SDF to MOL2 conversion")
        self.mgl_row = self._path_row(tools_box, "MGLTools python.exe/pythonsh.exe:", self.mgl_python, self._browse_mgl_python, 2, "Your MGLTools has python.exe; this replaces pythonsh.exe")
        self.prepare4_row = self._path_row(tools_box, "prepare_ligand4.py:", self.prepare_ligand4_path, self._browse_prepare4, 3, "AutoDockTools ligand preparation script")

        timeout_row = ttk.Frame(tools_box)
        timeout_row.grid(row=4, column=0, columnspan=4, sticky="ew", pady=8)
        ttk.Label(timeout_row, text="Timeout/ligand sec:", width=28).pack(side=tk.LEFT)
        ttk.Spinbox(timeout_row, from_=30, to=3600, textvariable=self.timeout_sec, width=10).pack(side=tk.LEFT)
        ttk.Button(timeout_row, text="Test selected tool paths", command=self._test_tool_paths).pack(side=tk.LEFT, padx=20)

        info = (
            "AutoDock Vina mode uses only Vina Meeko Python.exe.\n"
            "AutoDock4 mode uses OpenBabel obabel.exe + MGLTools python.exe/pythonsh.exe + prepare_ligand4.py.\n"
            "Your confirmed AutoDock4 test worked with MGLTools python.exe, so pythonsh.exe is not mandatory on your PC."
        )
        ttk.Label(tools_box, text=info, foreground="#004c99", justify=tk.LEFT).grid(row=5, column=0, columnspan=4, sticky="w", pady=15)

    def _path_row(self, parent, label, variable, browse_cmd, row, note=""):
        ttk.Label(parent, text=label, width=30).grid(row=row, column=0, sticky="w", pady=6)
        entry = ttk.Entry(parent, textvariable=variable, width=110)
        entry.grid(row=row, column=1, sticky="ew", padx=5, pady=6)
        btn = ttk.Button(parent, text="Browse", command=browse_cmd)
        btn.grid(row=row, column=2, sticky="w", pady=6)
        ttk.Label(parent, text=note, foreground="#555555").grid(row=row, column=3, sticky="w", padx=8, pady=6)
        parent.grid_columnconfigure(1, weight=1)
        return (entry, btn)

    def _build_logs_tab(self):
        frame = self.tab_logs
        summary = ttk.LabelFrame(frame, text="Running status", padding=10)
        summary.pack(fill=tk.X, pady=5)

        grid = ttk.Frame(summary)
        grid.pack(fill=tk.X)
        self._status_label(grid, "Current", self.current_var, 0, 0)
        self._status_label(grid, "Total", self.total_var, 0, 2)
        self._status_label(grid, "Success", self.success_var, 0, 4)
        self._status_label(grid, "Failed", self.failed_var, 0, 6)
        self._status_label(grid, "Duplicates", self.duplicate_var, 0, 8)

        ttk.Label(summary, text="Current molecule:").pack(anchor="w", pady=(8, 0))
        ttk.Label(summary, textvariable=self.current_name_var, foreground="#004c99").pack(anchor="w")

        self.progress = ttk.Progressbar(summary, orient="horizontal", mode="determinate")
        self.progress.pack(fill=tk.X, pady=8)

        btns = ttk.Frame(summary)
        btns.pack(fill=tk.X)
        ttk.Button(btns, text="Open live log window", command=self._open_log_window).pack(side=tk.LEFT, padx=3)
        ttk.Button(btns, text="Save visible log as TXT", command=self._save_log_as_txt).pack(side=tk.LEFT, padx=3)
        ttk.Button(btns, text="Open output folder", command=self._open_output_folder).pack(side=tk.LEFT, padx=3)

        log_box = ttk.LabelFrame(frame, text="Live log", padding=5)
        log_box.pack(fill=tk.BOTH, expand=True, pady=5)
        self.log_text = ScrolledText(log_box, height=24, wrap=tk.WORD)
        self.log_text.pack(fill=tk.BOTH, expand=True)
        self.log_widgets.append(self.log_text)

    def _status_label(self, parent, label, var, r, c):
        ttk.Label(parent, text=f"{label}:").grid(row=r, column=c, sticky="e", padx=(8, 2))
        ttk.Label(parent, textvariable=var, font=("Segoe UI", 10, "bold")).grid(row=r, column=c + 1, sticky="w", padx=(0, 12))

    def _build_reports_tab(self):
        frame = self.tab_reports
        out_box = ttk.LabelFrame(frame, text="Output", padding=10)
        out_box.pack(fill=tk.X, pady=5)
        row = ttk.Frame(out_box)
        row.pack(fill=tk.X, pady=3)
        ttk.Label(row, text="Output folder:", width=16).pack(side=tk.LEFT)
        ttk.Entry(row, textvariable=self.output_dir).pack(side=tk.LEFT, fill=tk.X, expand=True, padx=5)
        ttk.Button(row, text="Browse", command=self._browse_output).pack(side=tk.LEFT, padx=3)
        ttk.Button(row, text="Open", command=self._open_output_folder).pack(side=tk.LEFT, padx=3)

        note = (
            "After processing, this output folder will contain only:\n"
            "PDBQT/        -> final prepared ligand .pdbqt files\n"
            "ligands.txt   -> successful ligand names, one name per line\n\n"
            "Prepared SDF/MOL2 working files are temporary and are deleted automatically.\n"
            "No ZIP file is created."
        )
        ttk.Label(frame, text=note, justify=tk.LEFT).pack(anchor="w", pady=15)

    def _build_bottom_bar(self, parent):
        bar = ttk.Frame(parent, padding=(0, 8, 0, 0))
        bar.pack(fill=tk.X)
        self.run_btn = ttk.Button(bar, text="RUN LIGAND PREPARATION", command=self._start_run)
        self.run_btn.pack(side=tk.LEFT, padx=4)
        ttk.Button(bar, text="Go to Live Logs tab", command=lambda: self.nb.select(self.tab_logs)).pack(side=tk.LEFT, padx=4)
        ttk.Button(bar, text="Open live log window", command=self._open_log_window).pack(side=tk.LEFT, padx=4)
        ttk.Button(bar, text="Open output folder", command=self._open_output_folder).pack(side=tk.LEFT, padx=4)
        self.bottom_status = ttk.Label(bar, text="Ready", foreground="#004c99")
        self.bottom_status.pack(side=tk.RIGHT, padx=8)

    def _update_input_visibility(self):
        mode = self.input_mode.get()
        if mode == "SMILES text":
            self.input_path_row.pack_forget()
            self.csv_row.pack_forget()
            self.smiles_row.pack(fill=tk.X, pady=3)
        elif mode == "CSV with SMILES":
            self.input_path_row.pack(fill=tk.X, pady=3)
            self.csv_row.pack(fill=tk.X, pady=3)
            self.smiles_row.pack_forget()
        else:
            self.input_path_row.pack(fill=tk.X, pady=3)
            self.csv_row.pack_forget()
            self.smiles_row.pack_forget()

    def _update_mode_visibility(self):
        mode = self.docking_mode.get()
        if mode == "AutoDock Vina":
            for widget in self.vina_row:
                widget.configure(state="normal")
            for widget in self.obabel_row + self.mgl_row + self.prepare4_row:
                widget.configure(state="disabled")
        else:
            for widget in self.vina_row:
                widget.configure(state="disabled")
            for widget in self.obabel_row + self.mgl_row + self.prepare4_row:
                widget.configure(state="normal")

    def _browse_input_path(self):
        mode = self.input_mode.get()

        if mode == "CSV with SMILES":
            p = filedialog.askopenfilename(
                title="Select CSV file",
                filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
            )
            if p:
                self.input_files = []
                self.input_path.set(p)

        elif mode == "Structure files":
            # IMPORTANT: "All files" is first so Windows/Tk always displays
            # SDF, MOL, MOL2 and PDB together and allows mixed multi-selection.
            ps = filedialog.askopenfilenames(
                title="Select ligand structure files (SDF / MOL / MOL2 / PDB)",
                filetypes=[
                    ("All files", "*.*"),
                    ("SDF files", "*.sdf"),
                    ("MOL files", "*.mol"),
                    ("MOL2 files", "*.mol2"),
                    ("PDB files", "*.pdb"),
                ],
            )
            if ps:
                supported = [
                    str(Path(p))
                    for p in ps
                    if Path(p).suffix.lower() in SUPPORTED_STRUCTURE_EXTS
                ]
                unsupported = [
                    Path(p).name
                    for p in ps
                    if Path(p).suffix.lower() not in SUPPORTED_STRUCTURE_EXTS
                ]

                if unsupported:
                    messagebox.showwarning(
                        "Unsupported files ignored",
                        "These files were ignored because only SDF, MOL, MOL2 and PDB are supported:\n\n"
                        + "\n".join(unsupported)
                    )

                if not supported:
                    messagebox.showerror(
                        "No supported structure selected",
                        "Please select at least one .sdf, .mol, .mol2 or .pdb ligand file."
                    )
                    return

                self.input_files = supported
                self.input_path.set("; ".join(supported))

        elif mode == "Folder of structures":
            p = filedialog.askdirectory(
                title="Select folder containing SDF/MOL/MOL2/PDB ligand files"
            )
            if p:
                self.input_files = []
                self.input_path.set(p)

    def _browse_output(self):
        p = filedialog.askdirectory(title="Select output folder")
        if p:
            self.output_dir.set(p)

    def _browse_vina_python(self):
        p = filedialog.askopenfilename(title="Select Python.exe with Meeko installed", filetypes=[("Python executable", "python.exe"), ("Executable", "*.exe"), ("All files", "*.*")])
        if p:
            self.vina_python.set(p)

    def _browse_obabel(self):
        p = filedialog.askopenfilename(title="Select obabel.exe", filetypes=[("Executable", "*.exe"), ("All files", "*.*")])
        if p:
            self.obabel_path.set(p)

    def _browse_mgl_python(self):
        p = filedialog.askopenfilename(title="Select MGLTools python.exe or pythonsh.exe", filetypes=[("Executable", "*.exe"), ("All files", "*.*")])
        if p:
            self.mgl_python.set(p)

    def _browse_prepare4(self):
        p = filedialog.askopenfilename(title="Select prepare_ligand4.py", filetypes=[("Python file", "*.py"), ("All files", "*.*")])
        if p:
            self.prepare_ligand4_path.set(p)

    def _open_output_folder(self):
        path = Path(self.output_dir.get()).resolve()
        path.mkdir(parents=True, exist_ok=True)
        try:
            if sys.platform.startswith("win"):
                os.startfile(path)
            elif sys.platform == "darwin":
                subprocess.Popen(["open", str(path)])
            else:
                subprocess.Popen(["xdg-open", str(path)])
        except Exception as e:
            messagebox.showerror("Open folder failed", str(e))

    def _open_log_window(self):
        if self.log_window is not None and self.log_window.winfo_exists():
            self.log_window.lift()
            return
        self.log_window = tk.Toplevel(self.root)
        self.log_window.title("Live Ligand Preparation Log")
        self.log_window.geometry("950x620")
        top = ttk.Frame(self.log_window, padding=6)
        top.pack(fill=tk.BOTH, expand=True)
        text = ScrolledText(top, wrap=tk.WORD)
        text.pack(fill=tk.BOTH, expand=True)
        text.insert(tk.END, "".join(self.log_buffer))
        text.see(tk.END)
        self.log_widgets.append(text)

        def on_close():
            try:
                self.log_widgets.remove(text)
            except ValueError:
                pass
            self.log_window.destroy()
        self.log_window.protocol("WM_DELETE_WINDOW", on_close)

    def _save_log_as_txt(self):
        p = filedialog.asksaveasfilename(title="Save log as TXT", defaultextension=".txt", filetypes=[("Text files", "*.txt"), ("All files", "*.*")])
        if not p:
            return
        Path(p).write_text("".join(self.log_buffer), encoding="utf-8")
        messagebox.showinfo("Saved", f"Log saved:\n{p}")

    def log(self, msg):
        line = str(msg) + "\n"
        self.log_buffer.append(line)

        def _append():
            for widget in list(self.log_widgets):
                try:
                    widget.insert(tk.END, line)
                    widget.see(tk.END)
                except Exception:
                    pass
            self.root.update_idletasks()
        try:
            self.root.after(0, _append)
        except Exception:
            print(msg)

    def set_progress(self, current, total, name=""):
        def _set():
            self.progress["maximum"] = max(total, 1)
            self.progress["value"] = current
            self.current_var.set(str(current))
            self.total_var.set(str(total))
            if name:
                self.current_name_var.set(name)
            self.bottom_status.config(text=f"Running: {current}/{total}")
            self.root.update_idletasks()
        try:
            self.root.after(0, _set)
        except Exception:
            pass

    def set_counters(self, success, failed, duplicates):
        def _set():
            self.success_var.set(str(success))
            self.failed_var.set(str(failed))
            self.duplicate_var.set(str(duplicates))
        try:
            self.root.after(0, _set)
        except Exception:
            pass

    def _collect_settings(self):
        input_files = list(self.input_files)
        if self.input_mode.get() == "Structure files":
            # Rebuild from the visible path field when necessary so selected
            # structure files are never lost between GUI actions.
            visible_paths = [
                p.strip()
                for p in self.input_path.get().split(";")
                if p.strip()
            ]
            if visible_paths:
                input_files = visible_paths
        return {
            "docking_mode": self.docking_mode.get(),
            "input_mode": self.input_mode.get(),
            "input_path": self.input_path.get().strip(),
            "input_files": input_files,
            "smiles_text": self.smiles_text.get().strip(),
            "smiles_col": self.smiles_col.get().strip(),
            "name_col": self.name_col.get().strip(),
            "output_dir": self.output_dir.get().strip(),
            "force_field": self.force_field.get(),
            "num_conformers": self.num_conformers.get(),
            "random_seed": self.random_seed.get(),
            "timeout_sec": self.timeout_sec.get(),
            "remove_salts": self.remove_salts.get(),
            "canonical_tautomer": self.canonical_tautomer.get(),
            "deduplicate": self.deduplicate.get(),
            "vina_python": self.vina_python.get().strip(),
            "obabel_path": self.obabel_path.get().strip(),
            "mgl_python": self.mgl_python.get().strip(),
            "prepare_ligand4_path": self.prepare_ligand4_path.get().strip(),
        }

    def _validate_settings(self, settings):
        if not settings["output_dir"]:
            raise ValueError("Output folder is required")
        if settings["input_mode"] != "SMILES text" and not settings["input_path"]:
            raise ValueError("Input path is required")
        if settings["input_mode"] == "SMILES text" and not settings["smiles_text"]:
            raise ValueError("SMILES text is empty")

        if settings["input_mode"] == "Structure files":
            files = [Path(p) for p in settings.get("input_files", []) if p]
            if not files:
                raise ValueError("Select at least one SDF, MOL, MOL2 or PDB ligand file")
            missing = [str(p) for p in files if not p.exists()]
            if missing:
                raise ValueError("Selected structure file not found:\n" + "\n".join(missing))
            unsupported = [p.name for p in files if p.suffix.lower() not in SUPPORTED_STRUCTURE_EXTS]
            if unsupported:
                raise ValueError(
                    "Unsupported structure file(s):\n"
                    + "\n".join(unsupported)
                    + "\n\nSupported: SDF, MOL, MOL2, PDB"
                )

        if settings["input_mode"] == "Folder of structures":
            folder = Path(settings["input_path"])
            if not folder.exists() or not folder.is_dir():
                raise ValueError("Selected structure folder does not exist")
            supported_files = [
                p for p in folder.iterdir()
                if p.is_file() and p.suffix.lower() in SUPPORTED_STRUCTURE_EXTS
            ]
            if not supported_files:
                raise ValueError("No SDF, MOL, MOL2 or PDB ligand file found in the selected folder")

        if settings["docking_mode"] == "AutoDock Vina":
            if not settings["vina_python"] or not Path(settings["vina_python"]).exists():
                raise ValueError("Vina mode needs valid Python.exe where Meeko is installed")
        else:
            if not settings["mgl_python"] or not Path(settings["mgl_python"]).exists():
                raise ValueError("AutoDock4 mode needs valid MGLTools python.exe/pythonsh.exe path")
            if not settings["prepare_ligand4_path"] or not Path(settings["prepare_ligand4_path"]).exists():
                raise ValueError("AutoDock4 mode needs valid prepare_ligand4.py path")
            if not settings["obabel_path"] or not Path(settings["obabel_path"]).exists():
                raise ValueError("AutoDock4 mode needs valid OpenBabel obabel.exe path")

    def _test_tool_paths(self):
        settings = self._collect_settings()
        self.nb.select(self.tab_logs)
        self.log("\nTesting selected tool paths...")

        if Path(settings["vina_python"]).exists():
            ok, out, err, code = run_cmd([settings["vina_python"], "-m", "meeko.cli.mk_prepare_ligand", "--help"], timeout=30)
            self.log(f"Vina Meeko Python: {'OK' if ok else 'FAILED'} | {settings['vina_python']}")
            if not ok:
                self.log(f"  Error: {err.strip() or out.strip()}")
        else:
            self.log(f"Vina Meeko Python: FILE NOT FOUND | {settings['vina_python']}")

        if Path(settings["obabel_path"]).exists():
            ok, out, err, code = run_cmd([settings["obabel_path"], "-V"], timeout=30)
            self.log(f"OpenBabel: {'OK' if ok else 'FAILED'} | {settings['obabel_path']}")
            if out.strip():
                self.log("  " + out.strip())
            if err.strip():
                self.log("  " + err.strip())
        else:
            self.log(f"OpenBabel: FILE NOT FOUND | {settings['obabel_path']}")

        if Path(settings["mgl_python"]).exists() and Path(settings["prepare_ligand4_path"]).exists():
            ok, out, err, code = run_cmd([settings["mgl_python"], settings["prepare_ligand4_path"], "-h"], timeout=30)
            self.log(f"AutoDockTools prepare_ligand4: {'OK' if ok else 'FAILED'}")
            self.log(f"  MGL Python: {settings['mgl_python']}")
            self.log(f"  prepare_ligand4.py: {settings['prepare_ligand4_path']}")
            if not ok:
                self.log(f"  Error: {err.strip() or out.strip()}")
        else:
            self.log("AutoDockTools prepare_ligand4: FILE NOT FOUND for MGL Python or prepare_ligand4.py")

        self.log("Tool path test finished.")

    def _start_run(self):
        if self.is_running:
            messagebox.showinfo("Already running", "Processing is already running.")
            return
        try:
            settings = self._collect_settings()
            self._validate_settings(settings)
        except Exception as e:
            messagebox.showerror("Settings error", str(e))
            return

        self.is_running = True
        self.run_btn.configure(state="disabled")
        self.bottom_status.config(text="Running...")
        self.progress["value"] = 0
        self.current_var.set("0")
        self.total_var.set("0")
        self.success_var.set("0")
        self.failed_var.set("0")
        self.duplicate_var.set("0")
        self.nb.select(self.tab_logs)
        self.log("\nStarting ligand preparation...")
        self.log(f"Mode: {settings['docking_mode']}")
        self.log(f"Output: {settings['output_dir']}")

        def worker():
            try:
                prep = LigandPreparator(settings, log_func=self.log, progress_func=self.set_progress, counters_func=self.set_counters)
                result = prep.process_all()
                self.root.after(
                    0,
                    lambda: messagebox.showinfo(
                        "Complete",
                        f"Successful: {result['success']}\n"
                        f"Failed: {result['failed']}\n"
                        f"Duplicates: {result['duplicates']}\n\n"
                        f"PDBQT folder:\n{result['pdbqt_dir']}\n\n"
                        f"Ligand names:\n{result['ligands_txt']}"
                    )
                )
            except Exception as e:
                self.log("\nFATAL ERROR:")
                self.log(str(e))
                self.log(traceback.format_exc())
                self.root.after(0, lambda: messagebox.showerror("Fatal error", str(e)))
            finally:
                def done():
                    self.is_running = False
                    self.run_btn.configure(state="normal")
                    self.bottom_status.config(text="Done")
                self.root.after(0, done)

        threading.Thread(target=worker, daemon=True).start()


def launch_ligand_prep_gui():
    """Launch the GUI window from VS Code, normal Python, or VS Code Jupyter."""
    if tk is None:
        raise RuntimeError(f"Tkinter import failed: {TK_IMPORT_ERROR}")
    if Chem is None:
        raise RuntimeError(f"RDKit import failed: {RDKit_IMPORT_ERROR}")
    root = tk.Tk()
    app = LigandPrepGUI(root)
    root.mainloop()
    return app


if __name__ == "__main__":
    launch_ligand_prep_gui()

[21:08:42] Initializing MetalDisconnector
[21:08:42] Running MetalDisconnector
[21:08:42] Initializing Normalizer
[21:08:42] Running Normalizer
[21:08:42] Initializing MetalDisconnector
[21:08:42] Running MetalDisconnector
[21:08:42] Initializing Normalizer
[21:08:42] Running Normalizer
[21:08:42] Running LargestFragmentChooser
[21:09:00] Initializing MetalDisconnector
[21:09:00] Running MetalDisconnector
[21:09:00] Initializing Normalizer
[21:09:00] Running Normalizer
[21:09:00] Initializing MetalDisconnector
[21:09:00] Running MetalDisconnector
[21:09:00] Initializing Normalizer
[21:09:00] Running Normalizer
[21:09:00] Running LargestFragmentChooser
[21:09:04] Initializing MetalDisconnector
[21:09:04] Running MetalDisconnector
[21:09:04] Initializing Normalizer
[21:09:04] Running Normalizer
[21:09:04] Initializing MetalDisconnector
[21:09:04] Running MetalDisconnector
[21:09:04] Initializing Normalizer
[21:09:04] Running Normalizer
[21:09:04] Running LargestFragmentChooser
[21:09:13]